**Imports and Secure Key Loading**

In [1]:
import os
import requests
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

# Load the environment variables from .env file
load_dotenv()

# Grab the key 'api_key' in .env file
API_KEY = os.getenv('api_key')

if not API_KEY:
    print("⚠️ WARNING: API Key not found. Check your .env file.")
else:
    print("✅ API Key loaded successfully!")

# The top 10 coins to track
COINS = [
    'bitcoin', 'ethereum', 'tether', 'ripple', 'binancecoin', 
    'usd-coin', 'solana', 'tron', 'dogecoin', 'hyperliquid'
]

# Calculate timestamps for exactly 4 days ago to right now
end_time = int(datetime.now(timezone.utc).timestamp())
start_time = int((datetime.now(timezone.utc) - timedelta(days=4)).timestamp())

print(f"Fetching hourly data from {start_time} to {end_time}")

✅ API Key loaded successfully!
Fetching hourly data from 1773369495 to 1773715095


**The Fetch Function (with authentication)**

In [2]:
def fetch_hourly_history(coin_id, start, end):
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart/range"
    
    # Passing the key securely via the recommended Header method
    headers = {
        "x-cg-demo-api-key": API_KEY
    }
    
    params = {
        'vs_currency': 'usd',
        'from': start,
        'to': end
    }
    
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status() # Fail fast if the API blocks us
    return response.json()

**Pulling the Data**

In [5]:
# The script initializes an empty list to store the extracted records.
all_records = []

print("Starting data pull (Price + Market Cap)...")

# Iterates through the predefined list of target cryptocurrencies.
for coin in COINS:
    print(f"Fetching {coin}...")
    data = fetch_hourly_history(coin, start_time, end_time)
    
    # Extracts the 'prices' and 'market_caps' arrays from the JSON response.
    # Defaults to empty lists to prevent errors if the payload is missing data.
    prices = data.get('prices', [])
    market_caps = data.get('market_caps', [])
    
    # Constructs a dictionary mapping each timestamp to its corresponding market cap.
    # This enables efficient O(1) lookups during the merging process.
    mcap_dict = {item[0]: item[1] for item in market_caps}
    
    # Iterates over the price data, formats the timestamp to UTC, 
    # and retrieves the matching market cap from the dictionary.
    for item in prices:
        timestamp_ms = item[0]
        price = item[1]
        
        all_records.append({
            'coin_id': coin,
            'timestamp': datetime.fromtimestamp(timestamp_ms / 1000.0, tz=timezone.utc),
            'price_usd': price,
            'market_cap': mcap_dict.get(timestamp_ms) 
        })
        
    # The script pauses execution for 3 seconds to ensure compliance 
    # with the CoinGecko API rate limit (30 calls per minute).
    time.sleep(3) 

print(f"\nSuccess! Fetched {len(all_records)} total rows.")

Starting data pull (Price + Market Cap)...
Fetching bitcoin...
Fetching ethereum...
Fetching tether...
Fetching ripple...
Fetching binancecoin...
Fetching usd-coin...
Fetching solana...
Fetching tron...
Fetching dogecoin...
Fetching hyperliquid...

Success! Fetched 960 total rows.


In [6]:
# Converts the list of dictionaries into a structured Pandas DataFrame.
df = pd.DataFrame(all_records)

# Sorts the dataset chronologically by timestamp, and then alphabetically by coin ID.
# Resets the index to maintain a clean sequence.
df = df.sort_values(by=['timestamp', 'coin_id']).reset_index(drop=True)

# Configures Pandas to display floating-point numbers with two decimal places.
# This suppresses scientific notation, ensuring large market cap values remain readable.
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Outputs the final shape of the DataFrame and previews the first 15 rows.
print(f"DataFrame Shape: {df.shape}")
display(df.head(15))

DataFrame Shape: (960, 4)


,coin_id,timestamp,price_usd,market_cap
0,ripple,2026-03-13 03:03:25.150000+00:00,1.40,85821630699.31
1,tron,2026-03-13 03:03:25.474000+00:00,0.29,27486799573.75
2,usd-coin,2026-03-13 03:03:26.986000+00:00,1.00,78810287975.53
3,ethereum,2026-03-13 03:03:31.639000+00:00,2106.20,254120992197.29
4,solana,2026-03-13 03:03:40.837000+00:00,89.13,50902504397.20
5,dogecoin,2026-03-13 03:03:40.960000+00:00,0.10,14714509025.88
6,binancecoin,2026-03-13 03:03:42.232000+00:00,659.80,89969461329.70
7,hyperliquid,2026-03-13 03:03:44.814000+00:00,38.01,9060667502.91
8,bitcoin,2026-03-13 03:03:47.841000+00:00,71157.30,1422843138598.94
9,tether,2026-03-13 03:03:48.403000+00:00,1.00,183975335341.88


📝 API Exploration & Validation Summary

Objective: To test the CoinGecko /market_chart/range endpoint, validate API authentication, and confirm the successful concurrent extraction of hourly price and market capitalization data for the target assets without exceeding rate limits.

Process:

    Secure Authentication: The Demo API key was loaded from a local .env file using python-dotenv and injected securely into the HTTP request headers (x-cg-demo-api-key).

    Time Window Calculation: Dynamic Unix timestamps (UTC) were generated for a 4-day window. This specific timeframe (< 90 days) was deliberately chosen to force the CoinGecko API to return hourly granularity rather than daily aggregates.

    Rate-Limited Extraction: The script iterated through the 10 target coins (BTC, ETH, SOL, etc.), executing individual API calls while enforcing a 3-second time.sleep() delay between requests to remain safely below the free tier limit of 30 calls/minute.

    Data Transformation & Mapping: The script parsed the nested JSON response, extracting both the prices and market_caps arrays. A dictionary mapping technique was utilized to accurately merge the market caps with their corresponding prices based on exact timestamps. The native millisecond timestamps were converted into readable UTC datetime objects, and the combined data was flattened into a structured Pandas DataFrame. Formatting was applied to suppress scientific notation for large numerical values.

Key Findings:

    Authentication: Header-based API key injection executed flawlessly.

    Granularity Validated: The output timestamps confirmed exact 1-hour intervals (e.g., 03:03:25, 04:02:32).

    Schema Definition: The raw JSON was successfully mapped to a clean, comprehensive relational schema containing coin_id (string), timestamp (datetime), price_usd (float), and market_cap (float).